In [4]:
import dspy

lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

A **DSPy module** is a building block for programs that use LMs.

- Each built-in module abstracts a **prompting technique** (like chain of thought or ReAct). Crucially, they are generalized to handle any signature.

- A DSPy module has **learnable parameters** (i.e., the little pieces comprising the prompt and the LM weights) and can be invoked (called) to process inputs and return outputs.

- Multiple modules can be composed into bigger modules (programs). DSPy modules are inspired directly by NN modules in PyTorch, but applied to LM programs.


In [5]:
sentence = (
    "it's a charming and often affecting journey."  # example from the SST-2 dataset.
)

# 1) Declare with a signature.
classify = dspy.Predict("sentence -> sentiment: bool")

# 2) Call with input argument(s).
response = classify(sentence=sentence)

# 3) Access the output.
print(response.sentiment)

True


In [6]:
# Note that all DSPY modules are built using dspy.Predict
print(type(dspy.Predict("sentence -> sentiment: bool")))
print(type(dspy.ChainOfThought("question -> answer", n=5)))

<class 'dspy.predict.predict.Predict'>
<class 'dspy.predict.chain_of_thought.ChainOfThought'>


## Getting many completions

In [7]:
question = "What's something great about the ColBERT retrieval model?"

# 1) Declare with a signature, and pass some config.
classify = dspy.ChainOfThought("question -> answer", n=5)

# 2) Call with input argument.
response = classify(question=question)

# 3) Access the outputs.
response.completions.answer

['One great aspect of the ColBERT retrieval model is its efficient dual-encoder architecture that allows for rapid retrieval and reduces computational costs while maintaining high accuracy through late interaction techniques.',
 'A great aspect of the ColBERT retrieval model is its efficient late interaction mechanism, which allows for fast and effective retrieval by computing similarities between queries and documents more selectively, thus optimizing both accuracy and computational efficiency.',
 'One great aspect of the ColBERT retrieval model is its efficient two-tower architecture that allows for fast and effective document retrieval, combining traditional and deep learning methods for high precision and recall.',
 'One great aspect of the ColBERT retrieval model is its efficient combination of dense and sparse retrieval techniques, enabling high accuracy and fast document retrieval in large-scale information retrieval tasks.',
 'A great thing about the ColBERT retrieval model is 

In [8]:
# This is the first one
print(f"Reasoning: {response.reasoning}")
print(f"Answer: {response.answer}")

print("-" * 50)
# We can index like this too
print(f"reasoning 0 : {response.completions[0].reasoning}")
print(f"answer 0 : {response.completions[0].answer}")

Reasoning: The ColBERT retrieval model is notable for its efficiency and effectiveness in handling large-scale information retrieval tasks. One of its greatest strengths is its ability to leverage a dual-encoder architecture that separates the representation of queries and documents, allowing for rapid retrieval using efficient vector similarity search. This design significantly reduces the computational burden compared to traditional dense retrieval methods, while still maintaining high accuracy. Furthermore, ColBERT employs late interaction techniques, which means that it can compute similarities between queries and document representations only when necessary, thus further optimizing the retrieval process.
Answer: One great aspect of the ColBERT retrieval model is its efficient dual-encoder architecture that allows for rapid retrieval and reduces computational costs while maintaining high accuracy through late interaction techniques.
-------------------------------------------------

## Types of Modules

### dspy.ProgramOfThought
runs Python programs to solve a problem. This module requires deno to be installed.

In [ ]:
# import os

# os.environ["PATH"] = "/teamspace/studios/this_studio/.deno/bin:" + os.environ["PATH"]

In [ ]:
# pot = dspy.ProgramOfThought("question -> answer: int")
# response = pot(
#     question="How many letters a in the following string? fgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograefgipawjrgiajerigpajeiorgjaepogrjopaegjpoaergkopakerpogapgkpoaegkproaekgproaawegfkeaowkgeorfakwfokawpfejapwijfiwagjripawgreagjpareiwgkoaekrgpoagpkgrpaeograe"
# )
# response

Prediction(
    reasoning="The code counts the occurrences of the letter 'a' in the provided string. The output of the code indicates that there are 96 instances of the letter 'a' in the string.",
    answer=96
)

### ReAct

In [10]:
def evaluate_math(expression: str) -> float:
    return dspy.PythonInterpreter({}).execute(expression)


def get_weather(city: str) -> str:
    return f"The temperature in {city} is 1500 degrees." # unrealistic to make sure it's using the function


react = dspy.ReAct("question -> answer: float", tools=[evaluate_math, get_weather])

pred = react(question="What is 9362158 divided by the weather in paris?")
print(pred.answer)

0.0


## Compose multiple modules into a bigger program: Multi Hop Search

In [ ]:
import dspy

lm = dspy.LM('fireworks_ai/llama-v3-8b-instruct', max_tokens=3000)
gpt4o = dspy.LM('openai/gpt-4o', max_tokens=3000)

dspy.configure(lm=lm)

In [33]:
lm("Hello word!")

NotFoundError: litellm.NotFoundError: NotFoundError: Fireworks_aiException - {"error":{"code":"NOT_FOUND","message":"Model not found, inaccessible, and/or not deployed","requestId":"chatcmpl-cdc3957fac93449393f741931e718cdb","param":"model"}}

## Tracking LLM Usage

In [27]:
dspy.settings.configure(track_usage=True)

In [30]:
pred = react(question="What is 9362158 divided by the weather in Cairo?")
print(pred.get_lm_usage())

{'openai/gpt-4o-mini': {'completion_tokens': 272, 'prompt_tokens': 2509, 'total_tokens': 2781, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': 0, 'image_tokens': 0}}}
